# Report Q2 2025 by week and month

*Data Analysis in Python · Topic 6.2*

The e-commerce team's quarterly review needs April to June 2025 by week and by month, and a smoothed line of daily revenue for the chart. Make sure no partial week ends up next to the full ones.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that builds the `daily` table: one row per day, with the dates as the index (topic 6.1.'s steps).
2. In the first cell marked ✏️, create:
   - `days_per_week`: the number of days of data in each Monday-to-Sunday week of the `daily` table, as a Series
   - the `weekly` table: the `daily` table totaled by Monday-to-Sunday week, with only the weeks that have all 7 days, and a `conversion_rate` column: each week's transactions over its sessions
3. In the second cell marked ✏️, create the `monthly` table: the `daily` table totaled by month, with each month labelled by its first day, and a `conversion_rate` column.
4. In the last cell marked ✏️, create:
   - `revenue_7d`: the 7-day rolling average of the `daily` table's revenue, as a Series
   - `best_week`: the label of the week in the `weekly` table with the most revenue
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `days_per_week` has 14 weeks: 6 days in the first, the week ending April 6, and 1 in the last, the week ending July 6. The `weekly` table has the 12 full weeks, from the week ending April 13 to the one ending June 29, and its first week has $41,292.79 of revenue and a conversion rate of 2.32%. The `monthly` table has three rows, labelled April 1, May 1 and June 1, and June's conversion rate is 2.85%. `revenue_7d` starts with six `NaN`s and is about 9,100.13 on June 30. `best_week` is the week ending June 29, 2025.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-04", "2025-05", "2025-06"]
START, END = "2025-04-01", "2025-06-30"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# One row per day, with the dates as the index (topic 6.1). Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
daily.shape

## Weeks

In [ ]:
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly["conversion_rate"] = weekly["transactions"] / weekly["sessions"]
days_per_week, weekly

## Months

In [ ]:
monthly = daily.resample("MS").sum()
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
monthly

## The trend

In [ ]:
revenue_7d = daily["revenue"].rolling(7).mean()
best_week = weekly["revenue"].idxmax()
revenue_7d.tail(), best_week

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    days = found.get("days_per_week")
    weekly = found.get("weekly")
    if not isinstance(days, pd.Series) or not isinstance(days.index, pd.DatetimeIndex):
        results.append((False, "days_per_week should be a Series with one row per week: daily[\"sessions\"].resample(\"W\").count()."))
    elif len(days) != 14 or days.iloc[0] != 6 or days.iloc[-1] != 1:
        results.append((False, f"days_per_week should have 14 weeks, with 6 days in the first and 1 in the last. It has {len(days)} weeks, with {days.iloc[0]} and {days.iloc[-1]} days. Count the days of the daily table: resample(\"W\").count()."))
    elif not isinstance(weekly, pd.DataFrame) or not isinstance(weekly.index, pd.DatetimeIndex):
        results.append((False, "weekly should be the daily table totaled by week: daily.resample(\"W\").sum(), with only the full weeks."))
    elif len(weekly) == 14:
        results.append((False, "The weekly table has 14 weeks: the partial ones are still in it. The first has 6 days and the last 1, so keep the weeks where days_per_week is 7."))
    elif len(weekly) != 12 or weekly.index[0] != pd.Timestamp("2025-04-13") or weekly.index[-1] != pd.Timestamp("2025-06-29"):
        results.append((False, f"The weekly table should have the 12 full weeks, from the week ending April 13 to the one ending June 29. It has {len(weekly)} rows, from {weekly.index[0]:%B %d} to {weekly.index[-1]:%B %d}."))
    elif "revenue" in weekly.columns and abs(weekly["revenue"].iloc[0] - 5898.97) < 0.01:
        results.append((False, "The weekly table's first week has $5,898.97 of revenue: that's the mean of its days. Total the weeks with sum()."))
    elif not {"sessions", "transactions", "revenue", "conversion_rate"} <= set(weekly.columns) or abs(weekly["revenue"].iloc[0] - 41292.79) > 0.01:
        results.append((False, f"The weekly table should have the columns sessions, transactions, revenue and conversion_rate, and its first week $41,292.79 of revenue. It has {list(weekly.columns)}."))
    elif abs(weekly["conversion_rate"].iloc[0] - 0.023140) < 0.000005:
        results.append((False, "The weekly table's conversion_rate is the mean of the days' rates. Divide each week's total transactions by its total sessions (topic 4.4)."))
    elif abs(weekly["conversion_rate"].iloc[0] - 0.023151) > 0.000005:
        results.append((False, f"The weekly table's conversion_rate should be each week's transactions over its sessions: 2.32% in the first week. It is {weekly['conversion_rate'].iloc[0]:.2%}."))
    else:
        results.append((True, "The weekly table has the 12 full weeks of the quarter, April 13 to June 29, without the 6-day and 1-day weeks at the edges."))

    monthly = found.get("monthly")
    if not isinstance(monthly, pd.DataFrame):
        results.append((False, "monthly isn't a DataFrame yet. Total the daily table by month: daily.resample(\"MS\").sum()."))
    elif not isinstance(monthly.index, pd.DatetimeIndex):
        results.append((False, f"The monthly table's labels are {list(monthly.index)[:3]}, not dates. Use daily.resample(\"MS\").sum(), so the months stay dates."))
    elif len(monthly) == 3 and monthly.index[0] == pd.Timestamp("2025-04-30"):
        results.append((False, "The monthly table's months are labelled by their last day. The brief asks for the first day: resample(\"MS\")."))
    elif len(monthly) != 3 or monthly.index[0] != pd.Timestamp("2025-04-01") or "conversion_rate" not in monthly.columns:
        results.append((False, f"The monthly table should have three rows, labelled April 1, May 1 and June 1, with a conversion_rate column. It has {len(monthly)} rows."))
    elif abs(monthly["conversion_rate"].iloc[-1] - 0.028543) > 0.000005:
        results.append((False, f"The monthly table's conversion_rate should be each month's transactions over its sessions: 2.85% in June. It is {monthly['conversion_rate'].iloc[-1]:.2%}."))
    else:
        results.append((True, "The monthly table has April, May and June, labelled by their first day: June's sale lifts its conversion rate to 2.85%."))

    trend = found.get("revenue_7d")
    best = found.get("best_week")
    if not isinstance(trend, pd.Series) or len(trend) != 91 or not isinstance(trend.index, pd.DatetimeIndex):
        results.append((False, "revenue_7d should be a Series with one value per day, 91 in all: daily[\"revenue\"].rolling(7).mean()."))
    elif trend.isna().sum() == 0:
        results.append((False, f"revenue_7d has no NaN: its first value, {trend.iloc[0]:,.2f}, is a mean of one day. Leave min_periods out, so the first six days are NaN."))
    elif abs(trend.loc["2025-06-30"] - 63700.89) < 0.01:
        results.append((False, "revenue_7d is the 7-day total, not the average: use mean() instead of sum()."))
    elif pd.isna(trend.loc["2025-06-30"]):
        results.append((False, "revenue_7d is NaN on June 30: it's centered on each day, so the last days have no full window. Leave center out, so each value is that day and the six before it."))
    elif trend.isna().sum() != 6 or abs(trend.loc["2025-06-30"] - 9100.127143) > 0.01:
        results.append((False, f"revenue_7d should be the mean of each day and the six before it: six NaNs, and about 9,100.13 on June 30. It's {trend.loc['2025-06-30']:,.2f}, with {trend.isna().sum()} NaNs."))
    elif pd.api.types.is_number(best) and abs(best - 64719.56) < 0.01:
        results.append((False, "best_week is the best week's revenue. The brief asks for its label: weekly[\"revenue\"].idxmax()."))
    elif best != pd.Timestamp("2025-06-29"):
        results.append((False, f"best_week should be the label of the weekly table's row with the most revenue: weekly[\"revenue\"].idxmax(). It is {best!r}."))
    else:
        results.append((True, "Revenue's 7-day average peaks at the end of June, and the best week is the one ending June 29, the summer sale's first full week."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly["conversion_rate"] = weekly["transactions"] / weekly["sessions"]
days_per_week, weekly
```

```python
monthly = daily.resample("MS").sum()
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
monthly
```

```python
revenue_7d = daily["revenue"].rolling(7).mean()
best_week = weekly["revenue"].idxmax()
revenue_7d.tail(), best_week
```

</details>